In [1]:
import json
from pathlib import Path

processed_dir = Path("../data/processed")

# Load image descriptions
with open(processed_dir / "image_descriptions.json", encoding="utf-8") as f:
    image_data = json.load(f)

# Load markdown documents
markdown_docs = [
    {
        "paper_id": file.stem,
        "content": file.read_text(encoding="utf-8")
    }
    for file in processed_dir.glob("*.md")
]

print(f"Images: {len(image_data)}")
print(f"Markdown files: {len(markdown_docs)}")

Images: 55
Markdown files: 10


In [2]:
print(markdown_docs[0]["paper_id"])
print(markdown_docs[0]["content"][:3000])

2608.24921v1
## post-graph-rag: A PostgreSQL-Native Graph RAG Engine with Extraction-Time Quality Gates and a Temporal Relation Model

## Chandan Rajah

∗

Independent Researcher

Graph-based retrieval-augmented generation connects facts that no single passage states, but current implementations pay for that three times: in infrastructure, requiring a vector store, graph database and document store to be kept consistent; in graph quality, because an extraction pipeline that never refuses output fills the graph with edges that assert nothing; and over time, because a graph that only accumulates treats superseded and current facts alike.

post-graph-rag is an open-source engine addressing all three. Text chunks with embeddings, a canonical entity graph and community summaries live in one PostgreSQL database, with pgvector for search and edge tables for traversal. Extraction-time invariants run before anything is written: vague predicates, pronominal names and bare quantities are rejected

In [13]:
import re

def split_sections(text, paper_id):
    parts = re.split(r"(?=^##\s)", text, flags=re.MULTILINE)

    sections = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        lines = part.split("\n")

        if lines[0].startswith("## "):
            section = lines[0][3:].strip()
            content = "\n".join(lines[1:]).strip()
        else:
            section = "front_matter"
            content = part

        if content:
            sections.append({
                "paper_id": paper_id,
                "section": section,
                "content": content
            })

    return sections

In [14]:
all_sections = []

for doc in markdown_docs:
    sections = split_sections(
        doc["content"],
        doc["paper_id"]
    )
    all_sections.extend(sections)

print("Total sections:", len(all_sections))

Total sections: 286


In [15]:
for section in all_sections[:10]:
    print(
        section["paper_id"],
        "|",
        section["section"]
    )

2608.24921v1 | Chandan Rajah
2608.24921v1 | 1 Introduction
2608.24921v1 | 2 Related Work
2608.24921v1 | 3.1 Overview
2608.24921v1 | 3.2 Data model
2608.24921v1 | 3.3 Two-level tenancy
2608.24921v1 | 3.4 Concurrency and write ordering
2608.24921v1 | 3.5 Fail-closed invariants
2608.24921v1 | 4.1 Chunking with document context
2608.24921v1 | 4.2 Gleaning


In [6]:
for section in sections:
    words = len(section["content"].split())
    print(f"{words:5} words | {section['section']}")

  288 words | Chandan Rajah
  921 words | 1 Introduction
  808 words | 2 Related Work
   83 words | 3.1 Overview
  278 words | 3.2 Data model
  268 words | 3.3 Two-level tenancy
  198 words | 3.4 Concurrency and write ordering
  333 words | 3.5 Fail-closed invariants
  163 words | 4.1 Chunking with document context
  365 words | 4.2 Gleaning
   95 words | 4.3 Validation gates
  207 words | 4.4 Entity granularity
  170 words | 4.5 Predicate normalisation and controlled vocabulary
  222 words | 4.6 Entity resolution
  163 words | 4.7 Corroboration and negation
  126 words | 5.1 Validity intervals
  189 words | 5.2 Supersession from document order
   66 words | 5.3 As-of retrieval
  185 words | 5.4 Idempotent re-indexing and dormancy
  613 words | 6 Communities
   37 words | 7.1 Modes
  481 words | 7.2 Query conditioning and expansion
  241 words | 7.3 Two-channel relation retrieval
  298 words | 7.4 Synthesis
  191 words | 8.1 Setup
  489 words | 8.2 Contribution of each mechanism
  477 

In [7]:
paper_id = markdown_docs[0]["paper_id"]

sections_with_metadata = []

for section in sections:
    sections_with_metadata.append({
        "paper_id": paper_id,
        "section": section["section"],
        "content": section["content"]
    })

print(sections_with_metadata[0])

{'paper_id': '2608.24921v1', 'section': 'Chandan Rajah', 'content': '∗\n\nIndependent Researcher\n\nGraph-based retrieval-augmented generation connects facts that no single passage states, but current implementations pay for that three times: in infrastructure, requiring a vector store, graph database and document store to be kept consistent; in graph quality, because an extraction pipeline that never refuses output fills the graph with edges that assert nothing; and over time, because a graph that only accumulates treats superseded and current facts alike.\n\npost-graph-rag is an open-source engine addressing all three. Text chunks with embeddings, a canonical entity graph and community summaries live in one PostgreSQL database, with pgvector for search and edge tables for traversal. Extraction-time invariants run before anything is written: vague predicates, pronominal names and bare quantities are rejected; predicates are normalised onto an optional vocabulary; entities resolve to o

In [8]:
chunks = []

for section in sections_with_metadata:
    chunks.append({
        "paper_id": section["paper_id"],
        "section": section["section"],
        "content": section["content"]
    })

print(f"Total chunks: {len(chunks)}")

Total chunks: 36


In [9]:
for chunk in chunks[:5]:
    print("=" * 60)
    print(chunk["paper_id"])
    print(chunk["section"])
    print(chunk["content"][:200])

2608.24921v1
Chandan Rajah
∗

Independent Researcher

Graph-based retrieval-augmented generation connects facts that no single passage states, but current implementations pay for that three times: in infrastructure, requiring a
2608.24921v1
1 Introduction
Retrieval-augmented generation grounds a language model in retrieved text (Lewis et al., 2020; Karpukhin et al., 2020), and remains the default way to put a model to work over a private corpus. Its do
2608.24921v1
2 Related Work
Retrieval-augmented generation. Lewis et al. (2020) introduced RAG as joint training over a retriever and a generator; dense passage retrieval (Karpukhin et al., 2020) established the bi-encoder patte
2608.24921v1
3.1 Overview
Figure 1 shows the two paths through the system. Indexing chunks a document, extracts entities and triples with an LLM, validates that output, embeds chunks and entities, and writes vertices and edges
2608.24921v1
3.2 Data model
Vertex and edge tables, their audit and history shadows, an

In [10]:
for i, chunk in enumerate(chunks):
    chunk["chunk_id"] = i
    chunk["word_count"] = len(chunk["content"].split())

print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:10]:
    print(
        chunk["chunk_id"],
        "|",
        chunk["section"],
        "|",
        chunk["word_count"],
        "words"
    )

Total chunks: 36
0 | Chandan Rajah | 288 words
1 | 1 Introduction | 921 words
2 | 2 Related Work | 808 words
3 | 3.1 Overview | 83 words
4 | 3.2 Data model | 278 words
5 | 3.3 Two-level tenancy | 268 words
6 | 3.4 Concurrency and write ordering | 198 words
7 | 3.5 Fail-closed invariants | 333 words
8 | 4.1 Chunking with document context | 163 words
9 | 4.2 Gleaning | 365 words


In [11]:
print(image_data[0])

{'image_id': '2608.24921v1_figure_0.png', 'paper_id': '2608.24921v1', 'description': ' The image you\'ve provided appears to be a screenshot of a software application interface or possibly an educational diagram related to a search system. Here\'s a detailed description:\n\n1. Type of Visual: Architecture Diagram\n2. Main Purpose: To illustrate the architecture and components of a search system, potentially for a scientific research paper.\n3. Important Components:\n   - Textboxes: These likely represent the various modules or components of the search system. The labels are not clear, but they could be related to different aspects such as indexing, retrieval, and so on.\n   - Arrows and Lines: Indicate the flow of data or interactions between the components.\n4. Labels and Text: There is a label that reads "Indexing," suggesting that this is one of the stages within the search system\'s architecture. The exact content of other labels or text is not clear due to the resolution of the im

In [12]:
for doc in markdown_docs:
    print("=" * 60)
    print(doc["paper_id"])
    print(doc["content"][:500])

2608.24921v1
## post-graph-rag: A PostgreSQL-Native Graph RAG Engine with Extraction-Time Quality Gates and a Temporal Relation Model

## Chandan Rajah

∗

Independent Researcher

Graph-based retrieval-augmented generation connects facts that no single passage states, but current implementations pay for that three times: in infrastructure, requiring a vector store, graph database and document store to be kept consistent; in graph quality, because an extraction pipeline that never refuses output fills the gra
2608.24977v2
<!-- image -->

## Retrieved But Not Reliable: A Survey on Attacks, and Defenses in Retrieval-Augmented Generation

Minh Tran 1,2 , Cuong Dang 3 , Tuc Nguyen 4 , Khanh-Tung Tran 5 , Minh Huynh Nguyen 6 , Trinh Chau 7 , Kien Le 8 , Do Xuan Long 6 , Jiahao Zhang 9 , Fali Wang 9 , Hoang D. Nguyen 5 , Thanh Le 1,2 , Suhang Wang 9, †

1 Faculty of Information Technology, University of Science, Ho Chi Minh City, Vietnam 2 Vietnam National University, Ho Chi Minh City, Vietna

In [16]:
section_lengths = [
    (len(section["content"].split()), section["paper_id"], section["section"])
    for section in all_sections
]

section_lengths.sort(reverse=True)

for words, paper_id, section in section_lengths[:15]:
    print(f"{words:5} words | {paper_id} | {section}")

 5149 words | 2608.24977v2 | References
 2739 words | 2608.25986v1 | References
 2678 words | 2608.25717v1 | C.4 Distraction Context
 2352 words | 2608.25123v1 | 3 SelfGraphRAG
 1979 words | 2608.25618v1 | References
 1948 words | 2608.26379v1 | References
 1776 words | 2608.25717v1 | C.2 Perfect Context
 1543 words | 2608.26379v1 | 5 Evidence-Aware System Decisions
 1312 words | 2608.24977v2 | E Challenges and Future Directions
 1271 words | 2608.25466v1 | References
 1239 words | 2608.25986v1 | 4.1 Multi-Granularity Textual Context
 1206 words | 2608.26379v1 | 7 Does Better Evidence Improve RAG?
 1172 words | 2608.26194v1 | 7. References
 1137 words | 2608.24977v2 | C Variants of RAG
 1131 words | 2608.25466v1 | 4.1 Hybrid Retrieval Performance


In [17]:
!pip install -q langchain-text-splitters


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Ki_OJT\scientific-research-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [20]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [21]:
final_chunks = []

for section in all_sections:

    # Bỏ References khỏi retrieval
    if section["section"].lower() == "references":
        continue

    text = section["content"]

    # Section ngắn → giữ nguyên
    if len(text.split()) <= 1000:
        final_chunks.append(section)

    # Section dài → recursive split
    else:
        split_texts = splitter.split_text(text)

        for i, chunk_text in enumerate(split_texts):
            final_chunks.append({
                "paper_id": section["paper_id"],
                "section": section["section"],
                "content": chunk_text,
                "chunk_index": i
            })

print("Final chunks:", len(final_chunks))

Final chunks: 319


In [22]:
for chunk in final_chunks[:10]:
    print("=" * 60)
    print(chunk["paper_id"])
    print(chunk["section"])
    print(len(chunk["content"].split()), "words")
    print(chunk["content"][:300])

2608.24921v1
Chandan Rajah
288 words
∗

Independent Researcher

Graph-based retrieval-augmented generation connects facts that no single passage states, but current implementations pay for that three times: in infrastructure, requiring a vector store, graph database and document store to be kept consistent; in graph quality, because an
2608.24921v1
1 Introduction
921 words
Retrieval-augmented generation grounds a language model in retrieved text (Lewis et al., 2020; Karpukhin et al., 2020), and remains the default way to put a model to work over a private corpus. Its dominant failure mode is well documented and structural rather than incidental: retrieval returns pass
2608.24921v1
2 Related Work
808 words
Retrieval-augmented generation. Lewis et al. (2020) introduced RAG as joint training over a retriever and a generator; dense passage retrieval (Karpukhin et al., 2020) established the bi-encoder pattern that most production systems still use. Surveys (Gao et al., 2023) and failure ana